In [ ]:
import math
import torch
import time
import random
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.parameter import Parameter
from torch.nn import init
from torch import Tensor
from scipy.special import gamma 
import matplotlib.pyplot as plt

In [ ]:
# import math
# import torch
from torch.optim import Optimizer


class AdaMod(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), beta3=0.999, eps=1e-8, weight_decay=0):
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= eps:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 0: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 1: {betas[1]}")
        if not 0.0 <= beta3 < 1.0:
            raise ValueError(f"Invalid beta3 parameter: {beta3}")

        defaults = dict(lr=lr, betas=betas, beta3=beta3, eps=eps, weight_decay=weight_decay)
        super(AdaMod, self).__init__(params, defaults)

    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad.data
                if grad.is_sparse:
                    raise RuntimeError("AdaMod does not support sparse gradients")

                state = self.state[p]


                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p.data)
                    state["exp_avg_sq"] = torch.zeros_like(p.data)
                    state["exp_avg_lr"] = torch.zeros_like(p.data)

                exp_avg, exp_avg_sq, exp_avg_lr = state["exp_avg"], state["exp_avg_sq"], state["exp_avg_lr"]
                beta1, beta2 = group["betas"]
                beta3 = group["beta3"]

                state["step"] += 1


                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                bias_correction1 = 1 - beta1 ** state["step"]
                bias_correction2 = 1 - beta2 ** state["step"]
                denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(group["eps"])

                step_size = group["lr"] / bias_correction1

                step_size_tensor = step_size / denom


                exp_avg_lr.mul_(beta3).add_(step_size_tensor, alpha=1 - beta3)
                final_lr = torch.min(step_size_tensor, exp_avg_lr)

                if group["weight_decay"] != 0:
                    p.data.add_(p.data, alpha=-group["weight_decay"] * group["lr"])


                p.data.addcmul_(exp_avg, final_lr, value=-1.0)

        return loss


In [47]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class Fractional_Order_Matrix_Differential_Solver(torch.autograd.Function):
    @staticmethod
    def forward(ctx,input1,w,b,alpha,k,epoch):
        alpha = torch.tensor(alpha)
        k = torch.tensor(k)
        epoch = torch.tensor(epoch)
        ctx.save_for_backward(input1,w,b,alpha,k,epoch)
        outputs = input1@w + b
        return outputs

    @staticmethod
    def backward(ctx, grad_outputs):
        input1,w,b,alpha,k,epoch = ctx.saved_tensors
        x_fractional, w_fractional = Fractional_Order_Matrix_Differential_Solver.Fractional_Order_Matrix_Differential_Linear(input1,w,b,alpha,k,epoch)   
        x_grad = torch.mm(grad_outputs,x_fractional)
        w_grad = torch.mm(w_fractional,grad_outputs)
        b_grad = grad_outputs.sum(dim=0)
        return x_grad, w_grad, b_grad,None,None,None

    @staticmethod
    def Fractional_Order_Matrix_Differential_Linear(x,w,b,alpha,k,epoch):
        #w
        wf = w[:,0].view(1,-1)
        #main
        w_main = torch.mul(x,(torch.abs(wf)+1e-8)**(1-alpha)/gamma(2-alpha))
        #partial
        x_rows, x_cols = x.size()
        bias = torch.full((x_rows, x_cols),b[0].item())
        bias = bias.to(device)
        w_partial = torch.mul(torch.mm(x,wf.T).view(-1,1).expand(-1,x_cols) - torch.mul(x,wf) + bias, torch.sign(wf)*(torch.abs(wf)+1e-8)**(-alpha)/gamma(1-alpha))
        return w.T, (w_main + torch.exp(-k*epoch)*w_partial).T

class FLinear(nn.Module):
    
    __constants__ = ['in_features', 'out_features']
    in_features: int
    out_features: int
    weight: Tensor

    def __init__(self, in_features: int, out_features: int, alpha=0.9, k = 0.9, bias: bool = True,
                 device=None, dtype=None) -> None:
        factory_kwargs = {'device': device, 'dtype': dtype}
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.alpha = alpha
        self.k = k

        self.weight = Parameter(torch.empty((in_features, out_features), **factory_kwargs))
        if bias:
            self.bias = Parameter(torch.empty(out_features, **factory_kwargs))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            init.uniform_(self.bias, -bound, bound)

    def forward(self, x, epoch):
        return Fractional_Order_Matrix_Differential_Solver.apply(x, self.weight, self.bias, self.alpha, self.k, epoch)

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}"
    
def split(X,y):
    X_train,X_temp,y_train,y_temp = train_test_split(X,y,test_size=0.3,shuffle=False)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.333,shuffle=False)
    return X_train,X_val,X_test,y_train,y_val,y_test

#Mean Square Error
def MSE(pred,true):
    return np.mean((pred-true)**2)

#Mean Absolute Error
def MAE(pred, true):
    return np.mean(np.abs(pred-true))

def RMSE(pred,true):
    return np.sqrt(np.mean((pred-true)**2))

def MAPE(pred, true):
    return np.mean(np.abs((pred - true) / true))


In [53]:
slide_windows_size = 192  #i.e.,input length 192
pred_length = 384     #i.e.,prediction lengths 384
stock = 'ETTh2'    #ETTh2,DJI
df_DJIA = pd.read_csv(r'./data/'+stock+'.csv')
del df_DJIA['date']        #ETT2
# del df_DJIA['Date']        #DJI
scaler = MinMaxScaler(feature_range=(0, 1))

sca_DJIA = scaler.fit_transform(df_DJIA)

features_j = 6     #ETTh2:6,DJI:4
def create_sequences(data, slide_windows_size, pred_length):
    X, y = [], []
    for i in range(len(data) - slide_windows_size - pred_length + 1):
        X.append(data[i:i+slide_windows_size, :])  # sliding window size [seq_len, features]
        y.append(data[i+slide_windows_size:i+slide_windows_size+pred_length, features_j])  
    return np.array(X), np.array(y)

X, y = create_sequences(sca_DJIA, slide_windows_size, pred_length)
X = torch.Tensor(X).to(device)
y = torch.Tensor(y).to(device)

X_train,X_val,X_test,y_train,y_val,y_test = split(X,y)   #7:2:1 

In [54]:
alpha = 1.0   
k = 0.01      #In integer order, k does not play a role.

lrs =[0.01,0.005,0.001,0.0005]                     
weight_decays =[0.1,0.01,0.001,0.0001]                 

num_feature = 7     #ETTh1:7,DJI:5
batch_size = 256
set_seed()
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

class MLP(nn.Module):
    def __init__(self, input_size, hidden_size1=256, hidden_size2=128,output_size=pred_length):   
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear1 = FLinear(input_size, hidden_size1, alpha, k)  
        self.leakrelu1 = nn.LeakyReLU()                          
        self.linear2 = FLinear(hidden_size1, hidden_size2, alpha, k) 
        self.leakrelu2 = nn.LeakyReLU()
        self.linear3 = FLinear(hidden_size2, output_size, alpha, k)   

    def forward(self, x, epoch=0):
        x = self.flatten(x)    # (batch_size, seq_len*num_features)
        x = self.leakrelu1(self.linear1(x, epoch)) 
        x = self.leakrelu2(self.linear2(x, epoch))
        x = self.linear3(x, epoch)
        return x

lr_best = 0
weight_decay_best = 0
best_evaluation = 1000000

for lr in lrs:
    for weight_decay in weight_decays:
        set_seed()
        model = MLP(input_size=slide_windows_size*num_feature).to(device)
        num_epochs = 100   #
        best_loss = 1000000
        criterion = nn.MSELoss()
        optimizer = AdaMod(model.parameters(), lr=lr,weight_decay=weight_decay)
        for ii in range(num_epochs):
            model.train()
            loss_sum = 0
            for inputs, targets in train_loader:
                optimizer.zero_grad()
                outputs = model(inputs,ii)
                loss = criterion(outputs, targets)
                loss_sum += loss
                loss.backward()   #The default value of retain_graph is False.
                optimizer.step()
            # train_loss10.append(loss_sum.cpu().detach().numpy())     ###########
            
            # print(f"Epoch {ii + 1}/{num_epochs}, Train Loss: {loss_sum.cpu().detach().numpy():.4f}")
                
            model.eval()
            with torch.no_grad():
                Val_outputs = model(X_val)
                MSE_val = MSE(y_val.cpu().detach().numpy(),Val_outputs.cpu().detach().numpy())
                
                # val_loss10.append(MSE_val)   ########################Validation_loss
                
                # print(f"Epoch {ii + 1}/{num_epochs}, Val Loss: {MSE_val:.4f}")
                # print('')
                if best_loss > MSE_val:
                    best_loss = MSE_val
                    torch.save(model.state_dict(), r'./model/table_AdaMod/'+stock+'_model_fractional_'+str(alpha)+'_'+str(k)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_.pth')


        model.load_state_dict(torch.load('./model/table_AdaMod/'+stock+'_model_fractional_'+str(alpha)+'_'+str(k)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_.pth'))
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test)
        RMSE10 = RMSE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        MAE10 = MAE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        MAPE10 = MAPE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        print(str(alpha)+'_'+str(k)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'RMSE:{RMSE10:.4f}')
        print(str(alpha)+'_'+str(k)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'MAE:{MAE10:.4f}')
        print(str(alpha)+'_'+str(k)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'MAPE:{MAPE10:.4f}')
        if best_evaluation > RMSE10 + MAE10 + MAPE10:
            best_evaluation = RMSE10 + MAE10 + MAPE10
            print(str(alpha)+';'+str(k)+';'+str(lr)+';'+str(weight_decay))
            print(f'best_evaluation:{best_evaluation:.4f}')

1.0_0.01_0.01_0.1__RMSE:0.1131
1.0_0.01_0.01_0.1__MAE:0.0890
1.0_0.01_0.01_0.1__MAPE:0.1716
1.0;0.01;0.01;0.1
best_evaluation:0.3737
1.0_0.01_0.01_0.01__RMSE:0.1140
1.0_0.01_0.01_0.01__MAE:0.0909
1.0_0.01_0.01_0.01__MAPE:0.1668
1.0;0.01;0.01;0.01
best_evaluation:0.3717
1.0_0.01_0.01_0.001__RMSE:0.1197
1.0_0.01_0.01_0.001__MAE:0.0938
1.0_0.01_0.01_0.001__MAPE:0.1868
1.0_0.01_0.01_0.0001__RMSE:0.1218
1.0_0.01_0.01_0.0001__MAE:0.0954
1.0_0.01_0.01_0.0001__MAPE:0.1916
1.0_0.01_0.005_0.1__RMSE:0.1184
1.0_0.01_0.005_0.1__MAE:0.0927
1.0_0.01_0.005_0.1__MAPE:0.1837
1.0_0.01_0.005_0.01__RMSE:0.1170
1.0_0.01_0.005_0.01__MAE:0.0917
1.0_0.01_0.005_0.01__MAPE:0.1804
1.0_0.01_0.005_0.001__RMSE:0.1172
1.0_0.01_0.005_0.001__MAE:0.0919
1.0_0.01_0.005_0.001__MAPE:0.1810
1.0_0.01_0.005_0.0001__RMSE:0.1162
1.0_0.01_0.005_0.0001__MAE:0.0912
1.0_0.01_0.005_0.0001__MAPE:0.1787
1.0_0.01_0.001_0.1__RMSE:0.1047
1.0_0.01_0.001_0.1__MAE:0.0840
1.0_0.01_0.001_0.1__MAPE:0.1529
1.0;0.01;0.001;0.1
best_evaluation:0.3

In [55]:
alphas = [0.9,0.95,0.99,0.999]   
ks = [0.005,0.01,0.05,0.1,0.5,0.9]  

lr = 0.001                    
weight_decay = 0.001           

num_feature = 7     #ETTh1:7,DJI:5
batch_size = 256
set_seed()
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

class MLP(nn.Module):
    def __init__(self, input_size, hidden_size1=256, hidden_size2=128,output_size=pred_length):   
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear1 = FLinear(input_size, hidden_size1, alpha, k)  
        self.leakrelu1 = nn.LeakyReLU()                          
        self.linear2 = FLinear(hidden_size1, hidden_size2, alpha, k) 
        self.leakrelu2 = nn.LeakyReLU()
        self.linear3 = FLinear(hidden_size2, output_size, alpha, k)   

    def forward(self, x, epoch=0):
        x = self.flatten(x)    # (batch_size, seq_len*num_features)
        x = self.leakrelu1(self.linear1(x, epoch)) 
        x = self.leakrelu2(self.linear2(x, epoch))
        x = self.linear3(x, epoch)
        return x

lr_best = 0
weight_decay_best = 0
best_evaluation = 1000000

for alpha in alphas:
    for k in ks:
        set_seed()
        model = MLP(input_size=slide_windows_size*num_feature).to(device)
        num_epochs = 100   #
        best_loss = 1000000
        criterion = nn.MSELoss()
        optimizer = AdaMod(model.parameters(), lr=lr,weight_decay=weight_decay)
        for ii in range(num_epochs):
            model.train()
            loss_sum = 0
            for inputs, targets in train_loader:
                optimizer.zero_grad()
                outputs = model(inputs,ii)
                loss = criterion(outputs, targets)
                loss_sum += loss
                loss.backward()   #The default value of retain_graph is False.
                optimizer.step()
            # train_loss10.append(loss_sum.cpu().detach().numpy())     ###########
            
            # print(f"Epoch {ii + 1}/{num_epochs}, Train Loss: {loss_sum.cpu().detach().numpy():.4f}")
                
            model.eval()
            with torch.no_grad():
                Val_outputs = model(X_val)
                MSE_val = MSE(y_val.cpu().detach().numpy(),Val_outputs.cpu().detach().numpy())
                
                # val_loss10.append(MSE_val)   ########################Validation_loss
                
                # print(f"Epoch {ii + 1}/{num_epochs}, Val Loss: {MSE_val:.4f}")
                # print('')
                if best_loss > MSE_val:
                    best_loss = MSE_val
                    torch.save(model.state_dict(), r'./model/table_AdaMod/'+stock+'_model_fractional_'+str(alpha)+'_'+str(k)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_.pth')


        model.load_state_dict(torch.load('./model/table_AdaMod/'+stock+'_model_fractional_'+str(alpha)+'_'+str(k)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_.pth'))
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test)
        RMSE10 = RMSE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        MAE10 = MAE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        MAPE10 = MAPE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        print(str(alpha)+'_'+str(k)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'RMSE:{RMSE10:.4f}')
        print(str(alpha)+'_'+str(k)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'MAE:{MAE10:.4f}')
        print(str(alpha)+'_'+str(k)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'MAPE:{MAPE10:.4f}')
        if best_evaluation > RMSE10 + MAE10 + MAPE10:
            best_evaluation = RMSE10 + MAE10 + MAPE10
            print(str(alpha)+';'+str(k)+';'+str(lr)+';'+str(weight_decay))
            print(f'best_evaluation:{best_evaluation:.4f}')

0.9_0.005_0.001_0.001__RMSE:0.1080
0.9_0.005_0.001_0.001__MAE:0.0875
0.9_0.005_0.001_0.001__MAPE:0.1558
0.9;0.005;0.001;0.001
best_evaluation:0.3512
0.9_0.01_0.001_0.001__RMSE:0.1021
0.9_0.01_0.001_0.001__MAE:0.0822
0.9_0.01_0.001_0.001__MAPE:0.1412
0.9;0.01;0.001;0.001
best_evaluation:0.3255
0.9_0.05_0.001_0.001__RMSE:0.0972
0.9_0.05_0.001_0.001__MAE:0.0778
0.9_0.05_0.001_0.001__MAPE:0.1386
0.9;0.05;0.001;0.001
best_evaluation:0.3136
0.9_0.1_0.001_0.001__RMSE:0.1084
0.9_0.1_0.001_0.001__MAE:0.0873
0.9_0.1_0.001_0.001__MAPE:0.1566
0.9_0.5_0.001_0.001__RMSE:0.1074
0.9_0.5_0.001_0.001__MAE:0.0870
0.9_0.5_0.001_0.001__MAPE:0.1543
0.9_0.9_0.001_0.001__RMSE:0.1082
0.9_0.9_0.001_0.001__MAE:0.0872
0.9_0.9_0.001_0.001__MAPE:0.1568
0.95_0.005_0.001_0.001__RMSE:0.0937
0.95_0.005_0.001_0.001__MAE:0.0748
0.95_0.005_0.001_0.001__MAPE:0.1333
0.95;0.005;0.001;0.001
best_evaluation:0.3018
0.95_0.01_0.001_0.001__RMSE:0.1083
0.95_0.01_0.001_0.001__MAE:0.0860
0.95_0.01_0.001_0.001__MAPE:0.1605
0.95_0.05_